**`US_curate_footprints`**

This pipeline was developed to build a footprint-level building inventory
for hurricane damage modeling in the United States.

It runs the complete recipe chain: ingest, harmonize, enrich, and curate.

Recipe: `US_footprint-cheer-2026.yaml`

File: `src/openplaces/recipes/US/_all/footprint/cheer/2026/US_footprint-cheer-2026.yaml`

# Configure

In [ ]:
import argparse

from openplaces.core.schema import AdminId
from openplaces.io.curator import curate
from openplaces.io.enricher import enrich
from openplaces.io.harmonizer import harmonize
from openplaces.io.ingester import ingest
from openplaces.recipe import find_entity_recipe_id, get_recipe_by_id
from openplaces.timing import get_timer

In [ ]:
parser = argparse.ArgumentParser(
    description='Ingest, harmonize, enrich, and curate footprints'
)
parser.add_argument(
    '--recipe_id',
    help='Curation recipe (e.g. "US_footprint-cheer-2026")',
)
parser.add_argument(
    '--admin_ids',
    help='Admin unit IDs to process (e.g. "US-NC-BS")',
    nargs='*',
)
parser.add_argument(
    '--reprocess',
    help='Reprocess admin IDs even if output already exists',
    action='store_true',
)
parser.add_argument(
    '--redownload',
    help='Redownload input data from original source',
    action='store_true',
)
parser.add_argument(
    '--keep_unzipped',
    help='Keep unzipped datasets in heap folder after processing',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = (
    '--recipe_id US_footprint-cheer-2026 '
    # Shallotte township, Brunswick, NC (CHEER pilot: hurricane/flood risk)
    '--admin_ids US-NC-BS-SH '
    # '--admin_ids US-NC-BS '
    # '--admin_ids US-MA-MI-SO '
    # '--admin_ids US-MA-SU '
    # '--admin_ids US-FL-AL '
    # '--admin_ids US-TX-JE '
    # '--reprocess '
    # '--redownload '
    # '--keep_unzipped '
    '--verbose'
)

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

In [ ]:
# Pretty-print recipes
from openplaces.utils import pretty_print

curation_recipe = get_recipe_by_id(args.recipe_id)
harmonization_recipe_id = curation_recipe['entity_recipe']
enrichment_recipe_ids = [
    spec['recipe_id']
    for step in curation_recipe['pipeline']
    for spec in step.get('recipes', [])
]

print('Curation recipe:')
pretty_print(curation_recipe)
print('\nHarmonization recipe:')
pretty_print(get_recipe_by_id(harmonization_recipe_id))
for enrichment_recipe_id in enrichment_recipe_ids:
    print('\nEnrichment recipe:')
    pretty_print(get_recipe_by_id(enrichment_recipe_id))

# Ingest precursor datasets

In [ ]:
curation_recipe = get_recipe_by_id(args.recipe_id)
harmonization_recipe_id = curation_recipe['entity_recipe']
enrichment_recipe_ids = [
    spec['recipe_id']
    for step in curation_recipe['pipeline']
    for spec in step.get('recipes', [])
]

In [ ]:
# Save keyword arguments that will be passed to all ingest functions
ingest_kwargs = {
    key: getattr(args, key)
    for key in [
        'reprocess',
        'redownload',
        'keep_unzipped',
        'verbose',
    ]
}

# Harmonize and curate run at the recipes' process level (admin level 3).
# Truncate finer-grained admin IDs (e.g. a township) to their county;
# ingest and enrich calls below take args.admin_ids directly: ingest
# self-truncates (keeping the requested level for image recipes), and
# enrich restricts image-based steps to the requested units.
process_admin_ids = list(
    dict.fromkeys(str(AdminId(*AdminId(a).levels[:3])) for a in args.admin_ids)
)

# Track stage runtimes; saved to the logs directory at the end of the run
timer = get_timer(
    'US_curate_footprints',
    admin_id=process_admin_ids[0] if len(process_admin_ids) == 1 else None,
    verbose=args.verbose,
    overwrite=True,
    recipe_id=args.recipe_id,
    admin_ids=args.admin_ids,
)

In [ ]:
# US admin boundaries (for allocating Microsoft footprints to counties)
ingest('US_admin-census-2021_admin3', **ingest_kwargs)
timer.mark('ingest US_admin-census-2021_admin3')

In [ ]:
# Global building footprints: OpenBuildingsMap (OBM)
ingest('footprint-obm-2025', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest footprint-obm-2025')

In [ ]:
# US building footprints: Microsoft
ingest('US_footprint-microsoft-v2', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_footprint-microsoft-v2')

In [ ]:
# State footprints: auto-discovered per state from args.admin_ids
state_groups = {}
for aid_str in args.admin_ids or []:
    aid = AdminId(aid_str)
    if aid.get_level() >= 2:
        state_id = str(AdminId(*aid.levels[:2]))
        state_groups.setdefault(state_id, []).append(aid_str)

for state_id, children in state_groups.items():
    recipe_id = find_entity_recipe_id(
        state_id, 'footprint', stage='ingest', silent=True
    )
    if recipe_id:
        ingest(recipe_id, admin_ids=children, **ingest_kwargs)
timer.mark('ingest state footprints')

In [ ]:
# US building footprints: FEMA USA Structures
ingest('US_footprint-fema-2023', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_footprint-fema-2023')

In [ ]:
# State parcels: auto-discovered per state from args.admin_ids
for state_id, children in state_groups.items():
    recipe_id = find_entity_recipe_id(state_id, 'parcel', stage='ingest', silent=True)
    if recipe_id:
        ingest(recipe_id, admin_ids=children, **ingest_kwargs)
timer.mark('ingest state parcels')

In [ ]:
# US building points: National Structure Inventory (NSI)
ingest('US_building-nsi-2022', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest US_building-nsi-2022')

In [ ]:
# Global dwelling points: Overture
ingest('dwelling-overture-2025', admin_ids=args.admin_ids, **ingest_kwargs)
timer.mark('ingest dwelling-overture-2025')

# Harmonize

In [ ]:
harmonize(
    harmonization_recipe_id,
    admin_ids=process_admin_ids,
    reprocess=args.reprocess,
    verbose=args.verbose,
)
timer.mark('harmonize')

# Ingest enrichment inputs

Image recipes save at admin level 4, so imagery is fetched only for the
requested admin unit(s) (e.g. one township). Enrichment below also
restricts its image input to the requested units — imagery cached for
other townships in the same county is left alone — and records the
covered units in the evidence file's parquet footer, so towns can be
enriched incrementally. Evidence exists only for footprints with
imagery; curation handles the rest.

In [ ]:
image_recipe_ids = [
    get_recipe_by_id(recipe_id).get('image_recipe')
    for recipe_id in enrichment_recipe_ids
]

for image_recipe_id in dict.fromkeys(image_recipe_ids):
    if image_recipe_id is None:
        continue
    ingest(image_recipe_id, admin_ids=args.admin_ids, **ingest_kwargs)
    timer.mark(f'ingest {image_recipe_id}')

# Enrich

In [ ]:
# Pass args.admin_ids (not process_admin_ids): admin IDs deeper than the
# recipe's process level restrict image-based steps to those units, while
# output is still written at the process level (county).
for enrichment_recipe_id in enrichment_recipe_ids:
    enrich(
        enrichment_recipe_id,
        admin_ids=args.admin_ids,
        entity_recipe_id=harmonization_recipe_id,
        reprocess=args.reprocess,
        verbose=args.verbose,
    )
    timer.mark(f'enrich {enrichment_recipe_id}')

# Curate

In [ ]:
curate(
    args.recipe_id,
    admin_ids=process_admin_ids,
    reprocess=True,
    # reprocess=args.reprocess,
    verbose=args.verbose,
)
timer.mark('curate')

In [ ]:
# Report and save stage runtimes (JSON in the logs directory)
timer.summary()
timer.save()

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Loop script

In [ ]:
# CHEER_ADMIN3_IDS = 'US-NC-BA US-NC-BT US-NC-BL US-NC-BS US-NC-CD US-NC-CE US-NC-CW US-NC-CM US-NC-CN US-NC-CU US-NC-CI US-NC-DE US-NC-DP US-NC-ED US-NC-FR US-NC-GT US-NC-GE US-NC-HL US-NC-HT US-NC-HD US-NC-HO US-NC-HE US-NC-JH US-NC-JN US-NC-LN US-NC-MR US-NC-NA US-NC-NE US-NC-NO US-NC-ON US-NC-PM US-NC-PU US-NC-PD US-NC-PQ US-NC-PI US-NC-RB US-NC-SP US-NC-SC US-NC-TY US-NC-WK US-NC-WR US-NC-WI US-NC-WY US-NC-WO'.split()
# CHEER_ADMIN3_IDS = 'US-TX-AA US-TX-AU US-TX-BE US-TX-BI US-TX-BK US-TX-CU US-TX-CA US-TX-CH US-TX-CO US-TX-DE US-TX-DU US-TX-FT US-TX-FR US-TX-GV US-TX-GI US-TX-HN US-TX-RR US-TX-HG US-TX-JA US-TX-JR US-TX-JE US-TX-JG US-TX-JW US-TX-KE US-TX-KL US-TX-LA US-TX-LT US-TX-LK US-TX-MD US-TX-NE US-TX-NU US-TX-OR US-TX-RF US-TX-SP US-TX-SR US-TX-TY US-TX-VI US-TX-WR US-TX-WH US-TX-WB US-TX-WN US-TX-WY'.split()

In [ ]:
# args_list_cheer = (
#     ['--recipe_id', args.recipe_id, '--admin_ids']
#     + CHEER_ADMIN3_IDS
#     + ['--verbose', '--reprocess']
# )
# args_list_cheer

In [ ]:
# test_script(*args_list_cheer)

# Aggregate output files

In [ ]:
# from openplaces.api import aggregate_files

# aggregate_files(
#     args.recipe_id,
#     admin_level=2,
#     output_dir='share',
#     admin_ids_to_aggregate=CHEER_ADMIN3_IDS,
#     keep_original=True,
#     verbose=True,
#     combined=True,
# )

# Profile disk usage

The image caches written by this notebook can grow large (tens of GB per
county). `openplaces.diagnostics` reports where the disk space goes and
can delete location-specific image caches.

In [ ]:
from openplaces import diagnostics

# Disk usage by admin unit and dataset across the configured data
# directories (core, external, heap, cache, out)
usage = diagnostics.profile_disk_usage(min_size_mb=50)
usage.head(15)

In [ ]:
# Image caches by location; delete with a county or township ID
# (dry_run=True only reports; pass dry_run=False to actually delete)
diagnostics.list_image_caches()
# diagnostics.delete_image_caches(['US-NC-BS'], dry_run=True)

In [ ]:
from openplaces import get_entities

footprints = get_entities(args.recipe_id, args.admin_ids)
footprints.sample(5).T